# Phase 1: anchor training

Replaces `e1_phase1_many_worlds.ipynb`. Produces the adapter that
`arm_c`, `ft` and `combined` all load, so this runs before any of them.

## What changed

**Anchor worlds now come from the frozen generator.** `anchors.py` had
its own world builder with `START, GOAL = "E", "F"` fixed and
deterministic transitions only. The graded seeds have 19 distinct
start/goal pairs and a stochastic sibling, so the adapter was trained on
a distribution the eval never draws from. `anchors_v22.py` calls
`make_pair` at seeds 1000+ instead, and takes the prompt text from
`run_pilot.context_block` / `ask_block`, so an anchor example is the same
surface as a graded payload. `test_anchors.py` asserts both: all gold
answers score correct through `ecpm_parser`, and the intro and ask text
are identical to the eval's.

**Anchors cover four conditions and both modes.** `silent_break`,
`hard_removal`, `irrelevant` for the changed half, `no_change` for the
other half, roughly evenly split deterministic and stochastic. The old
set was one condition, deterministic only, which is why running the
adapter on stochastic degradation was untested transfer.

**Preservation positives are repeated.** The queried set holds exactly
one changed pair of four, so at world-level balance the pair-level
positive rate is 12.5% and answering "nothing changed" everywhere scores
0.875. That is exactly what the current adapter does on all 64 graded and
all 16 held-out replies. `--preservation-repeat 3` lifts the realised
rate to about 19%. Whether that is enough is an empirical question the
sanity check at the bottom answers.

**Three guards that would have caught the August bugs.** The optimizer
step count is computed and asserted non-zero before training starts (the
old cell did `3 // 4 = 0`, clamped to 1, so 20 epochs meant 20 steps). No
gold answer may be truncated by `MAX_LEN`, because a truncated answer
silently masks the whole loss. And the training surface is
`ecpm_eval.SYSTEM` plus `apply_chat_template`, imported from the module
the eval uses, so train and test cannot drift apart.

In [ ]:
%pip install -q -U transformers peft bitsandbytes accelerate datasets

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
REPO_PATH  = "/kaggle/input/ecpm-repo/ecpm-main"
EVAL_PATH  = "/kaggle/input/ecpm-eval"
OUT_DIR    = "/kaggle/working"
ADAPTER_OUT = f"{OUT_DIR}/anchor_adapter_{MODEL_NAME.split('/')[-1]}"

N_WORLDS      = 40
K             = 5
STOCH_SHARE   = 0.5
PRES_REPEAT   = 3
FIRST_SEED    = 1000
HOLDOUT_SEED  = 2000      # sanity-check worlds, never trained on

EPOCHS        = 3
LR            = 1e-4
GRAD_ACCUM    = 8
MAX_LEN       = 1600
LORA_R, LORA_ALPHA = 16, 32
LORA_DROPOUT  = 0.0       # dropout + gradient checkpointing raises
                          # CheckpointError, see the environment note
TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj",
           "gate_proj", "up_proj", "down_proj"]

import sys, os, json, collections
sys.path.insert(0, EVAL_PATH)
import anchors_v22, ecpm_eval as E
rp = anchors_v22.load_env(REPO_PATH)
print("environment loaded")

## Build the anchor set

Seeds start at 1000 and `build_anchor_set` asserts they clear the
official range, so an anchor world cannot be a graded world.

In [ ]:
worlds = anchors_v22.build_anchor_set(
    rp, n_worlds=N_WORLDS, k=K, stochastic_share=STOCH_SHARE,
    first_seed=FIRST_SEED, preservation_changed_repeat=PRES_REPEAT)
examples = anchors_v22.to_examples(worlds)

summary = anchors_v22.summarise(worlds)
for k, v in summary.items():
    print(f"  {k}: {v}")
json.dump(worlds, open(f"{OUT_DIR}/anchor_worlds.json", "w"))

assert summary["distinct_start_goal"] > 5, (
    "anchors collapsed onto too few start/goal pairs; the old bug")
assert summary["stochastic_worlds"] > 0, "no stochastic anchors"

## Verify the gold before training on it

Every gold answer must score `correct` through `ecpm_parser`, and the
prompt surface must match the eval's. If either fails, stop: phase 1
would teach the model to produce answers the scorer marks wrong.

In [ ]:
import ecpm_parser as ep

bad, checked = collections.Counter(), 0
for w in worlds:
    sc = anchors_v22._scenario(w["seed"], w["condition"], K)
    rec = rp.build_record(sc, w["deterministic"])
    for it in w["items"]:
        p = it["probe"]
        if p not in ("detection", "localization", "preservation", "adaptation"):
            continue
        s = ep.run_probe(rec, p, it["gold"],
                         queried_pairs=(w["queried_pairs"]
                                        if p == "preservation" else None))["scored"]
        checked += 1
        if not (s.get("correct") is True or s.get("accuracy") == 1.0
                or s.get("is_optimal") is True):
            bad[p] += 1
print(f"gold answers scored: {checked}, failures: {dict(bad) or 'none'}")
assert not bad, "anchor gold does not score correct; do not train on this"

## Tokenize with the loss masked to the answer

The August adapter trained on raw tokenized text while inference went
through `apply_chat_template`, so it learned a distribution the model is
never in at test time. `SYSTEM` is imported from `ecpm_eval`, the same
constant the eval sends, so the two cannot drift.

`MAX_LEN` is asserted against the longest example. A truncated gold
answer leaves nothing behind the mask and the example contributes no
gradient, silently.

In [ ]:
import torch
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def encode(ex):
    prefix = tok.apply_chat_template(
        [{"role": "system", "content": E.SYSTEM},
         {"role": "user", "content": ex["prompt"]}],
        add_generation_prompt=True, tokenize=True)
    answer = tok(ex["gold"] + tok.eos_token,
                 add_special_tokens=False)["input_ids"]
    ids = prefix + answer
    labels = [-100] * len(prefix) + answer[:]     # loss only on the answer
    return {"input_ids": ids, "labels": labels,
            "n_prefix": len(prefix), "n_answer": len(answer)}

enc = [encode(e) for e in examples]
lengths = [len(x["input_ids"]) for x in enc]
longest = max(lengths)
print(f"{len(enc)} examples, token length min {min(lengths)} "
      f"median {sorted(lengths)[len(lengths)//2]} max {longest}")
print(f"answer tokens: min {min(x['n_answer'] for x in enc)} "
      f"max {max(x['n_answer'] for x in enc)}")
assert longest <= MAX_LEN, (
    f"longest example is {longest} tokens but MAX_LEN is {MAX_LEN}; "
    "raise MAX_LEN or gold answers get truncated")
assert all(any(l != -100 for l in x["labels"]) for x in enc), \
    "an example has no unmasked label tokens"

In [ ]:
from torch.utils.data import Dataset

class Anchors(Dataset):
    def __init__(self, rows):
        self.rows = rows
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        return {"input_ids": r["input_ids"], "labels": r["labels"]}

def collate(batch):
    n = max(len(b["input_ids"]) for b in batch)
    pad = tok.pad_token_id
    out = {"input_ids": [], "labels": [], "attention_mask": []}
    for b in batch:
        gap = n - len(b["input_ids"])
        out["input_ids"].append(b["input_ids"] + [pad] * gap)
        out["labels"].append(b["labels"] + [-100] * gap)
        out["attention_mask"].append([1] * len(b["input_ids"]) + [0] * gap)
    return {k: torch.tensor(v) for k, v in out.items()}

train_ds = Anchors(enc)
print(len(train_ds), "training examples")

## Model

In [ ]:
from transformers import (AutoModelForCausalLM, BitsAndBytesConfig,
                          Trainer, TrainingArguments)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.bfloat16,
                         bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb, device_map="auto")
# gradient checkpointing off: the recompute pass saves a different number
# of tensors than LoRA dropout expects and raises CheckpointError. This
# call is the only place that turns it off.
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False)
model.config.use_cache = False

model = get_peft_model(model, LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    bias="none", task_type="CAUSAL_LM", target_modules=TARGETS))
model.print_trainable_parameters()

## Step math, checked before training

The August run chunked one evidence string into 3 examples and ran
`num_train_epochs=20` with `gradient_accumulation_steps=4`. `3 // 4 = 0`,
clamped to 1, so 20 epochs produced 20 optimizer steps. The assert below
makes that failure loud.

In [ ]:
steps_per_epoch = max(1, len(train_ds) // GRAD_ACCUM)
total_steps = steps_per_epoch * EPOCHS
print(f"{len(train_ds)} examples / grad_accum {GRAD_ACCUM} "
      f"= {steps_per_epoch} steps per epoch x {EPOCHS} epochs "
      f"= {total_steps} optimizer steps")
assert len(train_ds) >= GRAD_ACCUM, (
    f"{len(train_ds)} examples with grad_accum {GRAD_ACCUM} rounds to zero "
    "steps per epoch; lower GRAD_ACCUM or add worlds")
assert total_steps >= 20, f"only {total_steps} optimizer steps"

args = TrainingArguments(
    output_dir=f"{OUT_DIR}/phase1",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    warmup_steps=max(1, total_steps // 20),   # warmup_ratio is gone in 5.x
    lr_scheduler_type="cosine",
    logging_steps=5,
    save_strategy="no",
    report_to=[],
    bf16=True,
    gradient_checkpointing=False,
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                  data_collator=collate)
result = trainer.train()
print(result.metrics)

## Save, with provenance

`adapter_config.json` is what the eval notebook asserts against, so the
base model name and dropout recorded here are what stop a 3B adapter
being loaded on a 1.5B base later.

In [ ]:
model.save_pretrained(ADAPTER_OUT)
tok.save_pretrained(ADAPTER_OUT)

prov = {"model": MODEL_NAME, "phase": 1,
        "anchor_worlds": len(worlds), "anchor_examples": len(examples),
        "first_seed": FIRST_SEED, "k": K,
        "stochastic_share": STOCH_SHARE,
        "preservation_repeat": PRES_REPEAT,
        "epochs": EPOCHS, "lr": LR, "grad_accum": GRAD_ACCUM,
        "optimizer_steps": total_steps,
        "lora": {"r": LORA_R, "alpha": LORA_ALPHA, "dropout": LORA_DROPOUT,
                 "targets": TARGETS},
        "final_loss": result.metrics.get("train_loss"),
        "anchor_summary": summary}
json.dump(prov, open(f"{ADAPTER_OUT}/phase1_provenance.json", "w"), indent=1)

cfg = json.load(open(f"{ADAPTER_OUT}/adapter_config.json"))
assert cfg["base_model_name_or_path"] == MODEL_NAME
assert cfg["lora_dropout"] == 0.0
print("saved to", ADAPTER_OUT)
print(json.dumps(prov, indent=1)[:600])

## Sanity check on held-out anchor worlds

Seeds 2000+, never trained on, so this is a format-and-transfer check
rather than a result. What it answers:

* does the adapter emit parseable JSON on all four probes
* does it still answer "nothing changed" on every preservation probe, the
  failure the repeat was meant to break
* does it hold up on stochastic worlds, which the old anchors never
  covered

It is not the graded evaluation. That is `armc_eval_v22.ipynb` on the
official payloads.

In [ ]:
import pandas as pd

hold = anchors_v22.build_anchor_set(
    rp, n_worlds=8, k=K, stochastic_share=STOCH_SHARE,
    first_seed=HOLDOUT_SEED, preservation_changed_repeat=1)
assert not ({w["seed"] for w in hold} & {w["seed"] for w in worlds})

model.eval()
def ask(messages, max_new_tokens):
    e = tok.apply_chat_template(messages, add_generation_prompt=True,
                                return_tensors="pt", return_dict=True).to(model.device)
    with torch.no_grad():
        o = model.generate(**e, max_new_tokens=max_new_tokens,
                           do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(o[0, e["input_ids"].shape[1]:], skip_special_tokens=True)

rows = []
for w in hold:
    sc = anchors_v22._scenario(w["seed"], w["condition"], K)
    rec = rp.build_record(sc, w["deterministic"])
    seen = set()
    for it in w["items"]:
        p = it["probe"]
        if p in seen or p not in ("detection", "localization",
                                  "preservation", "adaptation"):
            continue
        seen.add(p)
        raw = ask([{"role": "system", "content": E.SYSTEM},
                   {"role": "user", "content": it["prompt"]}],
                  E.MAX_NEW_TOKENS.get(p, 400))
        res = ep.run_probe(rec, p, raw,
                           queried_pairs=(w["queried_pairs"]
                                          if p == "preservation" else None))
        rows.append({"arm": "arm_c", "seed": w["seed"],
                     "condition": w["condition"],
                     "deterministic": w["deterministic"], "mode": "single",
                     "probe": p, "turn": 2, "prompt": it["prompt"], "raw": raw,
                     "bare_json": E.is_bare_json(raw), "target": w["target"],
                     "parsed": res["parsed"], "scored": res["scored"]})

E.attach(REPO_PATH)
tab = E.table(rows, arms=["arm_c"])
display(pd.DataFrame(tab)[["arm", "sens", "spec", "localize", "pres_acc",
                           "pres_const", "target_recall", "pres_parsed",
                           "route_valid", "route_optimal", "bare_json"]])

const = sum(1 for r in rows if r["probe"] == "preservation"
            and r["parsed"].get("status") == "ok"
            and not any(p["changed"] for p in r["parsed"]["pairs"]))
n_pres = sum(1 for r in rows if r["probe"] == "preservation")
print(f"\nall-unchanged preservation replies: {const}/{n_pres}")
if const == n_pres:
    print("still constant. Raise PRES_REPEAT, or accept that judgement "
          "preservation is unreportable for this arm and use "
          "belief_self_consistency from the two-turn run instead.")